# Lecture 19 - PyTorch and Scripting

This tutorial builds on the excellent how-to: https://docs.python.org/3/howto/argparse.html

### Overview

In this lecture we'll cover more advanced topics in Python programming. 

We'll start with a brief introduction to PyTorch, a popular deep learning library. 

We'll then move on to scripting in Python, which is a powerful way to write more complex programs.

 - Introduction to PyTorch
    - Tensors
    - Autograd
 - Scripting in Python
    - Introduce `__main__`
    - Argument parsing with argparse
 - Example script
 

## PyTorch

PyTorch is a popular deep learning library that is widely used in academia and industry. It is known for its flexibility and ease of use. PyTorch is built on top of the Torch library, which is written in C++.

PyTorch provides two main features: tensors and autograd. Tensors are similar to NumPy arrays, but they can be used on GPUs. Autograd is a system for automatic differentiation, which is used to compute gradients in neural networks.

Together, tensors and autograd make it easy to build and train deep learning models in PyTorch.

### Tensors

Tensors are multi-dimensional arrays that can be used to store data. They are similar to NumPy arrays, but they can be used on GPUs. Tensors can be created using the `torch.tensor` function.

In [ ]:
import torch

# Create a tensor
x = torch.tensor([[1, 2], [3, 4]])

print(x)


Tensor operations can be performed using the `torch` module. For example, we can add two tensors together using the `torch.add` function.

In [ ]:
# Create two tensors
x = torch.tensor([[1, 2], [3, 4]])
y = torch.tensor([[5, 6], [7, 8]])

# Add the two tensors together
z = torch.add(x, y)

print(z)


Tensors can be moved to the GPU using the `to` method. This allows us to perform computations on the GPU, which can be much faster than on the CPU.

```python
# Create a tensor
x = torch.tensor([[1, 2], [3, 4]])

# Move the tensor to the GPU
x = x.to('cuda')

print(x)
```


### Autograd

Autograd is a system for automatic differentiation, which is used to compute gradients in neural networks. Autograd works by keeping track of the operations that are performed on tensors, and then computing the gradients of those operations using the chain rule.

To use autograd, we need to set the `requires_grad` attribute of a tensor to `True`. This tells PyTorch to keep track of the operations that are performed on the tensor.

In [ ]:
# Create a tensor with requires_grad=True
x = torch.tensor([5.], requires_grad=True)
# Note, only floating point tensors can require gradients

print(x)



We can then perform operations on the tensor, and autograd will compute the gradients of those operations.


In [ ]:
# Perform an operation on the tensor
y = x * 2
print(y)


In [ ]:

# Compute the gradient of the operation
y.backward()

print(x.grad)


The `grad` attribute of a tensor contains the gradient of the tensor with respect to some output. We can use this gradient to update the parameters of a neural network using gradient descent.

### Simple Neural Network

Now that we've covered the basics of PyTorch, let's build a simple neural network. We'll create a neural network with one hidden layer and train it on the Iris dataset.


In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:

# Standardize the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert the data to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.int64)
y_test = torch.tensor(y_test, dtype=torch.int64)


In [ ]:
import torch.nn as nn
import torch.optim as optim

# Create a neural network
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.fc1 = nn.Linear(4, 10)
        self.fc2 = nn.Linear(10, 3)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = NeuralNetwork()


In [ ]:

# Train the neural network
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

for epoch in range(100):
    optimizer.zero_grad()
    output = model(X_train)
    loss = criterion(output, y_train)
    # The backward() method computes the gradient of the loss with respect to the model parameters (we don't need to compute the gradient manually)
    loss.backward()
    # The step() method updates the model parameters based on the gradient
    optimizer.step()
    # Output the loss every epoch
    print(f'Epoch {epoch + 1}, Loss: {loss.item()}')



In [ ]:

# Evaluate the neural network
## We use torch.no_grad() to disable gradient computation because we don't need it for evaluation
with torch.no_grad():
    output = model(X_test)
    _, predicted = torch.max(output, 1)
    accuracy = (predicted == y_test).sum().item() / y_test.size(0)

print(f'Accuracy: {accuracy}')


## Scripting

Notebooks are amazing for interactive scripting, data exploration and simple analysis. 

There are many occasions however when it's more useful to be able to script your analysis. For example:
 - Long running processes / analyses, such as training a deep learning model
 - Running multiple variations on an analysis
 - Creating a versatile tool for different analyses
 - Running with Kubernetes and other distributed computing platforms
 - Others??

### Execution

Let's take a look at a very simple script: [my_module.py](my_module.py).

We can run a python module like this from the command line easily:

In [ ]:
! python my_module.py

On linux and macOS machines we can specify that the file should be run with Python and execute it directly

We just need to add a 'shebang' at the top of our script: `!/usr/bin/env python3`

Now we can run it directly:

In [ ]:
! ./my_module_x.py

One problem with this approach is that if we import the module it will run that code every time. This probably isn't what we want...

In [ ]:
import my_module_x

Python uses a special name for a module when it is being executed directly and we can use this to check if the module is being executed or imported:

In [ ]:
if __name__ == '__main__':
    print("I'm in a script")

Let's take a look at [my_module_main.py](my_module_main.py). Now if we import the module we don't get the message printed:

In [ ]:
import my_module_main

In [ ]:
! ./my_module_main.py

### Command line arguments

This is fine for very simple scripts that always perfrom the same actions, but we usually want to be able to alter the behaviour each time we run the script. 

Let's take a look at an example command line program to get an idea of what that looks like: `ls`.

Python provides a very convenient method for managing and parsing command line arguments using the `argparse` library: [my_module_main_args.py](my_module_main_args.py)

In [ ]:
! ./my_module_main_args.py --help

This simple setup already provides some useful funcionality:
 - Includes a help description describing correct usage
 - Catches incorrect arguments

How do we add more useful arguments? And how do we access those?

There are a few options, but the builtin `argparse` library provides the most flexibility pretty easily:

In [ ]:
import argparse

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("echo")  # Declare a (required) argument called echo
args = parser.parse_args(['hello'])  # Pretend we just called the program like: my_module_main_args.py foo
print(args)

In [ ]:
args.echo

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("echo", help="echo the string you use here")
parser.print_help() # my_script.py --help

OK, let's try something a bit more useful:

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("square", help="display a square of a given number")
args = parser.parse_args(['2'])
print(args.square**2)

Let's fix that:

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("square", help="display a square of a given number",
                    type=int)
args = parser.parse_args(['2'])
print(args.square**2)

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("square", help="display a square of a given number",
                    type=int)
args = parser.parse_args(['four'])
print(args.square**2)

We can also include optional arguments using the '--' prefix:

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("--verbosity", help="increase output verbosity")
args = parser.parse_args([])
if args.verbosity:
    print("verbosity turned on")


In [ ]:
args = parser.parse_args(['--verbosity', '1'])
if args.verbosity:
    print("verbosity turned on")


Sometimes we might not want any particular value, but just a flag. We can do that using the `action` keyword:

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("--verbose", help="increase output verbosity",
                    action="store_true")
args = parser.parse_args(['--verbose'])
if args.verbose:
    print("verbosity turned on")

We can even provide a shorthand:

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("-v", "--verbose", help="increase output verbosity",
                    action="store_true")
args = parser.parse_args(['-v'])
if args.verbose:
    print("verbosity turned on")

Combining these arguments is easy:

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("square", type=int,
                    help="display a square of a given number")
parser.add_argument("-v", "--verbose", action="store_true",
                    help="increase output verbosity")

In [ ]:
args = parser.parse_args(['2'])
answer = args.square**2
if args.verbose:
    print(f"the square of {args.square} equals {answer}")
else:
    print(answer)


In [ ]:
args = parser.parse_args(['2', '-v'])
answer = args.square**2
if args.verbose:
    print(f"the square of {args.square} equals {answer}")
else:
    print(answer)


Here's an example of a (simple) complete script: [example_script.py](example_script.py)

We've include multiple arguments of a particular type and a 'countable' optional argument.

## Exercise 1.

Work in pairs to create a script that accepts multiple commands to:
 - Print the contents of a NetCDF file
 - Calculate the time average, and writes the result to an output file
 - Provide a useful error message if the file doesn't exist, or xarray can't open it

There is an example NetCDF file in the public DataHub directory.

Solution: [nc.py](nc.py)